# 08 | Publicação e geração operacional

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** disponibilizar versões aprovadas e gerar a proposta pela API. **Entradas:** campanha e decisões07. Por padrão consulta apenas; altere os controles explicitamente. **Esperado:** versão publicada distinta de residente.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.academic_ui import campaign_result
result=campaign_result()
DEPLOY_DEMO=os.getenv('V4_DEPLOY_DEMO','false').lower()=='true'
GENERATE=os.getenv('V4_GENERATE','false').lower()=='true'
GENERATION_KEY=os.getenv('V4_GENERATION_KEY','v4-notebook-proposal-final-003')
selectors=[{k:m[k] for k in ('model_name','model_version')} for m in result['models']]
if DEPLOY_DEMO:
    for index,selector in enumerate(selectors):
        call('POST',f"/v1/serving/models/{selector['model_name']}/deploy",dict(**selector,policy='PRELOAD' if index==0 else 'LAZY'),
             key=f"v4-notebook-deploy-{selector['model_name']}-{selector['model_version']}",operator=True)
display(pd.DataFrame(call('GET','/v1/serving/models')))
display(call('GET','/v1/serving/runtime'))


**Executar:** geração HTTP, aguardando job persistido. **Resultado:**70 ofertas/107 origens, `proposal_id` salvo automaticamente para09. Mesma chave recupera; nova chave gera nova proposta, sem sobrescrever anteriores.

In [ ]:

import time
if GENERATE:
    payload=dict(schema_version='poc-planilha-turmas-v2',snapshot_id=result['snapshot_id'],cycle_code='2024-2',proposal_revision=1,
                 model_selectors=selectors,requested_by='jupyter-poc')
    job=call('POST','/v2/proposals/generations',payload,key=GENERATION_KEY)
    for _ in range(90):
        job=call('GET',job['status_url'])
        if job['status'] in ('COMPLETED','FAILED'):
            break
        time.sleep(2)
    if job['status']!='COMPLETED':
        raise RuntimeError('Geração não concluída: '+str({k:job[k] for k in ('job_id','status','error_code')}))
    save_context(proposal_id=job['proposal_id'])
    display({k:job[k] for k in ('job_id','proposal_id','status','proposal_url')})
else:
    display('READ_ONLY: geração não solicitada. Use os controles após aprovação.')


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.